# Chapter 24 — Causal Inference

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/ch24_causal_inference.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## 24.1 Potential outcomes

## 24.2 The policy: the 56th GST Council meeting

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS
from pypdf import PdfReader

from indiadata import mospi_cpi, mospi_cpi_items

GST_URL = ("https://gstcouncil.gov.in/sites/default/files/2025-09/"
           "press_release_press_information_bureau_0.pdf")
GST_PDF = Path("data/raw/gst/gstcouncil_56_press_release.pdf")
if not GST_PDF.exists():
    GST_PDF.parent.mkdir(parents=True, exist_ok=True)
    GST_PDF.write_bytes(requests.get(GST_URL, timeout=60).content)

text = "\n".join(page.extract_text() for page in PdfReader(GST_PDF).pages)
text = re.sub(r"[ \t]+", " ", text).replace("ﬂ", "fl").replace("ﬁ", "fi")
print(text[:270])

In [ ]:
start = text.index("The changes in GST rates of all goods except")
print(text[start:start + 210])

In [ ]:
start2 = text.index("Annexure -II", text.index("Annexure-I"))  # the annexure,
annex2 = text[start2:text.index("Annexure-III")]          # not the mention of it
HEADING = re.compile(r"(\d+%|Nil)\s*to\s*(\d+%|Nil)", flags=re.IGNORECASE)


def gst_heading(phrase):
    """Rate-change heading under which `phrase` appears in Annexure II."""
    heading = None
    for line in annex2.splitlines():
        m = HEADING.search(line)
        if m and len(line.strip()) < 30:          # a heading, not a description
            heading = f"{m[1]} to {m[2]}"
        if phrase.lower() in line.lower():
            return heading
    return "not found"


for phrase in ["Hair oil, shampoo", "Toilet Soap", "Butter and other fats",
               "Chocolates", "Air-conditioning machines", "Television sets",
               "Motorcycles of engine capacity", "Bicycles and other cycles"]:
    print(f"{phrase:32s} {gst_heading(phrase)}")

## 24.3 From tariff headings to CPI items

In [ ]:
TREATED = {   # CPI item: (old rate %, new rate %, Annexure II/III line)
    "Butter": (12, 5, "Butter and other fats (i.e. ghee, butter oil, etc.)"),
    "Ghee": (12, 5, "Butter and other fats (i.e. ghee, butter oil, etc.)"),
    "Milk: Condensed/ Powder": (12, 5, "Condensed milk"),
    "Biscuits, Chocolates, etc.": (18, 5, "Pastry, cakes, biscuits; Chocolates"),
    "Ice-cream": (18, 5, "Ice cream and other edible ice"),
    "Papad, Bhujia, Namkeen, Mixture, Chanachur": (12, 5, "Namkeens, bhujia, mixture"),
    "Sauce, Jam, Jelly (gm)": (12, 5, "Sauces; Jams, fruit jellies"),
    "Pickles (gm)": (12, 5, "Vegetables ... preserved by vinegar"),
    "Sewai, Noodles": (12, 5, "Pasta ... noodles"),
    "Fruit Juice and Shake (litre)": (12, 5, "Fruit or nut juices"),
    "Mineral Water (litre)": (18, 5, "Waters, including natural or artificial mineral waters"),
    "Dates": (12, 5, "Dates (soft or hard) ... dried"),
    "Other Dry Fruits": (12, 5, "Other nuts, dried ... Almonds"),
    "Hair Oil, Shampoo, Hair Cream": (18, 5, "Hair oil, shampoo"),
    "Toilet Soap": (18, 5, "Toilet Soap (other than industrial soap)"),
    "Toothpaste, Toothbrush, Comb, etc.": (18, 5, "Dental floss, toothpaste; Tooth brushes"),
    "Shaving Cream, Aftershave Lotion": (18, 5, "Shaving cream, shaving lotion"),
    "Candle (no.)": (12, 5, "Candles, tapers and the like"),
    "Matches (box)": (12, 5, "All goods-safety matches"),
    "Stainless Steel Utensils": (12, 5, "household articles of iron & steel; Utensils"),
    "Other Metal Utensils": (12, 5, "household articles of copper / aluminium"),
    "Other Crockery & Utensils": (12, 5, "Tableware, kitchenware ... of porcelain or china"),
    "Sewing Machine": (12, 5, "Sewing machines"),
    "Bicycle(without Accessories)": (12, 5, "Bicycles and other cycles"),
    "Umbrella, Raincoat": (12, 5, "Umbrellas and sun umbrellas"),
    "Leather Boots, Shoes": (12, 5, "Footwear of sale value not exceeding Rs.2500"),
    "Leather Sandals, Chappals, etc.": (12, 5, "Footwear of sale value not exceeding Rs.2500"),
    "Rubber / Pvc Footwear": (12, 5, "Footwear of sale value not exceeding Rs.2500"),
    "Other Leather Footwear": (12, 5, "Footwear of sale value not exceeding Rs.2500"),
    "Other Footwear": (12, 5, "Footwear of sale value not exceeding Rs.2500"),
    "Sports Goods, Toys, etc.": (12, 5, "Sports goods; Toys like tricycles"),
    "Medicine (non-institutional)": (12, 5, "All Drugs and medicines"),
    "Spectacles": (12, 5, "Spectacles and goggles for correcting vision"),
    "Motor Cycle, Scooter": (28, 18, "Motorcycles ... not exceeding 350cc"),
    "Tyres & Tubes": (28, 18, "New pneumatic tyres, of rubber"),
    "Television": (28, 18, "Television sets (including LCD and LED)"),
    "Air Conditioner, Air Cooler": (28, 18, "Air-conditioning machines"),
    "Barber, Beautician, etc.": (18, 5, "Beauty and physical well-being services (Annex. III)"),
    "Hotel Lodging Charges": (12, 5, "Hotel accommodation <= Rs 7,500 a day (Annex. III)"),
}
treated_table = pd.DataFrame(TREATED, index=["old %", "new %", "source line"]).T
treated_table["old→new"] = (treated_table["old %"].astype(str) + " → "
                            + treated_table["new %"].astype(str))
print(f"{len(treated_table)} treated items")
treated_table["old→new"].value_counts()

In [ ]:
pd.set_option("display.max_colwidth", 60)
treated_table[["old→new", "source line"]]

In [ ]:
CONTROLS = {
    "Staples, rate unchanged (nil or 5%)": [
        "Rice – Other Sources", "Wheat/ Atta – Other Sources", "Maida",
        "Suji, Rawa", "Besan", "Arhar, Tur", "Moong", "Masur", "Urd",
        "Gram: Split", "Sugar - Other Sources", "Gur", "Salt",
        "Tea: Leaf (gm)", "Mustard Oil", "Groundnut Oil",
        "Refined Oil [sunflower, Soyabean, Saffola, Etc.]", "Coconut Oil",
        "Turmeric (gm)", "Jeera (gm)", "Dhania (gm)", "Black Pepper (gm)",
        "Dry Chillies (gm)", "Milk: Liquid (litre)", "Curd", "Eggs (no.)",
        "Chicken", "Goat Meat/mutton", "Fish, Prawn"],
    "Goods not in the annexures": [
        "Mobile Handset", "Refrigerator", "Washing Machine", "Electric Fan",
        "PC/ Laptop/ Other Peripherals Incl. Software",
        "Electric Bulb, Tubelight", "Bedstead", "Almirah, Dressing Table",
        "Chair, Stool, Bench, Table", "Lock", "Clock, Watch",
        "Books, Journals: First Hand", "Newspapers, Periodicals",
        "Sanitary Napkins"],
    "Outside GST": [
        "Petrol for Vehicle", "Diesel for Vehicle", "Electricity (std. Unit)",
        "Country Liquor (litre)", "Foreign/refined Liquor or Wine (litre)",
        "Beer (litre)"],
    "Tobacco (change deferred)": ["Cigarettes (no.)", "Bidi (no.)",
                                  "Leaf Tobacco (gm)"],
    "Services not in the annexures": [
        "Doctor’s/ Surgeon’s Fee-first Consultation (non-institutional)",
        "Tuition and Other Fees (school, College, Etc.)",
        "Private Tutor/ Coaching Centre", "Bus/tram Fare", "Railway Fare",
        "Taxi, Auto-rickshaw Fare", "Tailor", "Washerman, Laundry, Ironing",
        "Domestic Servant/cook", "Telephone Charges: Mobile",
        "Internet Expenses", "Monthly Charges for Cable Tv Connection",
        "House Rent, Garage Rent", "Cooked Meals Purchased (no.)"],
}
food_controls = CONTROLS.pop("Staples, rate unchanged (nil or 5%)")
control_items = [i for group in CONTROLS.values() for i in group]
print(f"{len(control_items)} control items:",
      {g: len(v) for g, v in CONTROLS.items()})
print(f"plus {len(food_controls)} food staples, kept aside for a robustness check")

In [ ]:
EXCLUDED = {
    "Motor Car, Jeep": "small cars 28→18, larger cars 28→40, and the compensation cess abolished",
    "Cold Beverages: Bottled/canned (litre)": "sugary drinks 28→40 (a tax rise), plain water 18→5",
    "Coal": "rate 5→18 but compensation cess removed",
    "Coffee: Powder (gm)": "instant coffee 18→5, roasted/ground coffee unchanged",
    "Prepared Sweets, Cake, Pastry": "cakes 18→5, sweets bought from shops unchanged",
    "Other Packaged Processed Food": "many headings, several rates",
    "Cashewnut": "roasted/salted 12→5, raw cashew unchanged",
    "Powder, Snow, Cream, Lotion and Perfume": "talcum powder 18→5, creams and perfume not listed",
    "Stationery, Photocopying Charges": "notebooks and pencils 12→nil, photocopying unchanged",
    "Shirts, T-shirts (no.) and other clothing": "5% up to Rs 2,500 (was 12% from Rs 1,000), 18% above",
    "Gold, Silver": "rate unchanged, but prices follow world bullion markets",
    "Fresh vegetables and fruit": "untreated but very volatile; used in Exercise 5",
}
pd.Series(EXCLUDED, name="reason for exclusion").to_frame()

## 24.4 The CPI item indices

In [ ]:
cpi = mospi_cpi_items(range(2014, 2026))
print(f"{len(cpi):,} rows, {cpi['item'].nunique()} items, "
      f"{cpi['date'].min():%b %Y} to {cpi['date'].max():%b %Y}")

missing = [i for i in list(TREATED) + control_items + food_controls
           if i not in set(cpi["item"])]
assert not missing, missing
print("All", len(TREATED) + len(control_items) + len(food_controls),
      "item names found")

In [ ]:
panel = cpi[cpi["item"].isin(list(TREATED) + control_items + food_controls)].copy()
panel["treated"] = panel["item"].isin(list(TREATED)).astype(int)
panel["lp"] = 100 * np.log(panel["index"])        # 100 × log price
panel = panel[panel["date"] >= "2024-09-01"]       # 12 months before the cut
food_panel = panel[panel["item"].isin(food_controls)]
panel = panel[~panel["item"].isin(food_controls)]  # main sample: no food staples
print(panel.groupby("treated")["item"].nunique().rename({0: "control", 1: "treated"}))
print("Months per item:", panel.groupby("item").size().unique())

In [ ]:
base = panel[panel["date"] == "2025-08-01"].set_index("item")["lp"]
panel["lp_rel"] = panel["lp"] - panel["item"].map(base)
avg = panel.groupby(["date", "treated"])["lp_rel"].mean().unstack()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(avg.index, avg[0], color="#9db7d5", lw=2.2, label="Control items (rate unchanged)")
ax.plot(avg.index, avg[1], color="#1f4e79", lw=2.2, label="Treated items (GST cut)")
ax.axvline(pd.Timestamp("2025-09-22"), color="#c55a11", ls="--", lw=1.2)
ax.text(pd.Timestamp("2025-09-10"), avg.min().min(), "GST cut\n22 Sep 2025",
        color="#c55a11", ha="right", va="bottom", fontsize=9)
ax.axhline(0, color="black", lw=0.6)
ax.set_ylabel("Log price relative to Aug 2025 (×100 ≈ %)")
ax.set_title("Prices of treated items fell after the GST cut", loc="left",
             fontsize=13)
ax.legend(frameon=False, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(rect=(0, 0.04, 1, 1))
fig.text(0.01, 0.005, "Unweighted mean across items. Source: author's "
         "calculation from MoSPI CPI item indices (base 2012 = 100), All India "
         "combined; GST Council, 56th meeting.", fontsize=8, color="grey")
plt.show()

## 24.5 Difference-in-differences by hand

In [ ]:
def period(d):
    if d == pd.Timestamp("2025-08-01"):
        return "before"
    if d >= pd.Timestamp("2025-10-01"):
        return "after"
    return None


two = panel.assign(period=panel["date"].map(period)).dropna(subset=["period"])
means = (two.groupby(["treated", "period"])["lp"].mean().unstack()
            [["before", "after"]].rename(index={0: "control", 1: "treated"}))
means["change"] = means["after"] - means["before"]
means.round(2)

In [ ]:
did = means.loc["treated", "change"] - means.loc["control", "change"]
print(f"Change, treated items: {means.loc['treated', 'change']:+.2f}")
print(f"Change, control items: {means.loc['control', 'change']:+.2f}")
print(f"Difference-in-differences: {did:+.2f} log points (≈ per cent)")

## 24.6 DiD as a regression with fixed effects

In [ ]:
reg = panel[panel["date"] != "2025-09-01"].copy()
reg["post"] = (reg["date"] >= "2025-10-01").astype(int)
reg["treat_post"] = reg["treated"] * reg["post"]
reg = reg.set_index(["item", "date"])

twfe = PanelOLS(reg["lp"], reg[["treat_post"]], entity_effects=True,
                time_effects=True).fit(cov_type="clustered", cluster_entity=True)
b, se = twfe.params["treat_post"], twfe.std_errors["treat_post"]
print(f"DiD (TWFE): {b:.2f}  clustered SE {se:.2f}  "
      f"95% CI [{b - 1.96 * se:.2f}, {b + 1.96 * se:.2f}]")
print(f"Items: {reg.index.get_level_values(0).nunique()}, "
      f"observations: {twfe.nobs:,}")

In [ ]:
naive = PanelOLS(reg["lp"], reg[["treat_post"]], entity_effects=True,
                 time_effects=True).fit()
print(f"SE, classical: {naive.std_errors['treat_post']:.2f};  "
      f"clustered by item: {se:.2f}")

## 24.7 Pass-through: how much of the tax cut reached prices?

In [ ]:
for old, new in [(12, 5), (18, 5), (28, 18)]:
    full = (1 + new / 100) / (1 + old / 100) - 1
    print(f"{old}% → {new}%: full pass-through = {full:+.2%}  "
          f"(log points: {100 * np.log(1 + full):+.2f})")

In [ ]:
old_rate = reg.index.get_level_values("item").map(
    {k: v[0] for k, v in TREATED.items()}).fillna(0).to_numpy()
new_rate = reg.index.get_level_values("item").map(
    {k: v[1] for k, v in TREATED.items()}).fillna(0).to_numpy()
rate = np.where(reg["post"] == 1, new_rate, old_rate)
reg["tau"] = 100 * np.log(1 + rate / 100)

pt = PanelOLS(reg["lp"], reg[["tau"]], entity_effects=True,
              time_effects=True).fit(cov_type="clustered", cluster_entity=True)
rho, rho_se = pt.params["tau"], pt.std_errors["tau"]
print(f"Pass-through elasticity: {rho:.2f} (clustered SE {rho_se:.2f}), "
      f"95% CI [{rho - 1.96 * rho_se:.2f}, {rho + 1.96 * rho_se:.2f}]")

In [ ]:
cut_of = {k: f"{v[0]}→{v[1]}" for k, v in TREATED.items()}
item_level = reg.index.get_level_values("item")
groups = {}
for g in ["12→5", "18→5", "28→18"]:
    name = "tp_" + g.replace("→", "_")
    reg[name] = (item_level.map(cut_of) == g).astype(int) * reg["post"].to_numpy()
    groups[g] = name
het = PanelOLS(reg["lp"], reg[list(groups.values())], entity_effects=True,
               time_effects=True).fit(cov_type="clustered", cluster_entity=True)
het_table = pd.DataFrame({
    "DiD (log pts)": het.params.to_numpy(),
    "SE": het.std_errors.to_numpy(),
    "full pass-through (log pts)": [100 * np.log((1 + int(g.split("→")[1]) / 100)
                                                 / (1 + int(g.split("→")[0]) / 100))
                                    for g in groups],
}, index=list(groups))
het_table["pass-through"] = (het_table["DiD (log pts)"]
                             / het_table["full pass-through (log pts)"])
het_table["items"] = pd.Series(cut_of).value_counts()
het_table.round(2)

In [ ]:
change = (two.groupby(["item", "period"])["lp"].mean().unstack()
             .eval("after - before"))
control_change = change[control_items].mean()
items = treated_table[["old %", "new %"]].astype(float).copy()
items["full (log pts)"] = 100 * np.log((1 + items["new %"] / 100)
                                       / (1 + items["old %"] / 100))
items["DiD (log pts)"] = change[items.index] - control_change
items["pass-through"] = items["DiD (log pts)"] / items["full (log pts)"]
items = items.sort_values("pass-through", ascending=False)
items.round(2)

In [ ]:
by_rate = (items.assign(cut=treated_table["old→new"])
                .groupby("cut")[["full (log pts)", "DiD (log pts)", "pass-through"]]
                .median())
by_rate["items"] = treated_table["old→new"].value_counts()
print("Median across items, by rate change:")
by_rate.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 9))
order = items.sort_values("DiD (log pts)", ascending=False)
y = np.arange(len(order))
ax.scatter(order["full (log pts)"], y, marker="|", s=120, color="#c55a11",
           label="Full pass-through")
ax.scatter(order["DiD (log pts)"], y, color="#1f4e79", s=25,
           label="Estimated (item DiD)")
ax.hlines(y, order["full (log pts)"], order["DiD (log pts)"],
          color="#9db7d5", lw=1)
ax.axvline(0, color="black", lw=0.6)
ax.set_yticks(y)
ax.set_yticklabels([s[:38] for s in order.index], fontsize=8)
ax.set_xlabel("Price change, Aug 2025 → Oct–Dec 2025, relative to controls "
              "(log points ×100)")
ax.set_title("Durables passed on more of the cut than toiletries and foods",
             loc="left", fontsize=12, pad=28)
ax.legend(frameon=False, loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2,
          fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(rect=(0, 0.03, 1, 1))
fig.text(0.01, 0.005, "Source: author's calculation from MoSPI CPI item "
         "indices and GST Council (56th meeting) Annexures II–III.",
         fontsize=8, color="grey")
plt.show()

In [ ]:
top = items.sort_values("DiD (log pts)").head(5)
print("Largest relative price falls:")
print(top[["full (log pts)", "DiD (log pts)", "pass-through"]].round(2).to_string())
print(f"\nTreated items whose price fell relative to controls: "
      f"{(items['DiD (log pts)'] < 0).sum()} of {len(items)}")

## 24.8 Parallel trends and the event study

In [ ]:
es = panel.copy()
months = sorted(es["date"].unique())
cut = months.index(pd.Timestamp("2025-09-01"))
es["k"] = es["date"].map({m: i - cut for i, m in enumerate(months)})
ks = [k for k in sorted(es["k"].unique()) if k != -1]
for k in ks:
    es[f"e{k}"] = ((es["k"] == k) & (es["treated"] == 1)).astype(int)
es = es.set_index(["item", "date"])
ev = PanelOLS(es["lp"], es[[f"e{k}" for k in ks]], entity_effects=True,
              time_effects=True).fit(cov_type="clustered", cluster_entity=True)
coef = pd.DataFrame({"k": ks, "beta": ev.params.to_numpy(),
                     "se": ev.std_errors.to_numpy()})
coef = pd.concat([coef, pd.DataFrame({"k": [-1], "beta": [0.0], "se": [0.0]})]
                 ).sort_values("k").set_index("k")
coef.loc[-6:].round(2).T

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
pre = coef.index < 0
ax.errorbar(coef.index[pre], coef["beta"][pre], yerr=1.96 * coef["se"][pre],
            fmt="o", color="#9db7d5", ecolor="#9db7d5", capsize=2, ms=4,
            label="Before the cut (leads)")
ax.errorbar(coef.index[~pre], coef["beta"][~pre], yerr=1.96 * coef["se"][~pre],
            fmt="o", color="#1f4e79", ecolor="#1f4e79", capsize=2, ms=5,
            label="After the cut")
ax.axhline(0, color="black", lw=0.6)
ax.axvline(-0.5, color="#c55a11", ls="--", lw=1)
ax.set_xlabel("Months relative to September 2025 (reference: August 2025 = −1)")
ax.set_ylabel("Treated − control, log points")
ax.set_title("Event study: flat before the cut, a modest fall after it",
             loc="left", fontsize=13)
ax.legend(frameon=False, loc="lower left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(rect=(0, 0.04, 1, 1))
fig.text(0.01, 0.005, "Item and month fixed effects; 95% intervals from SEs "
         "clustered by item. Source: author's calculation from MoSPI CPI item "
         "indices.", fontsize=8, color="grey")
plt.show()

In [ ]:
leads = [f"e{k}" for k in ks if k < -1]
R = np.array([[1.0 if name == c else 0.0 for name in ev.params.index]
              for c in leads])                  # one row per restriction
wald = ev.wald_test(restriction=R, value=np.zeros(len(leads)))
print(f"Joint test of {len(leads)} leads: stat = {wald.stat:.1f}, "
      f"p-value = {wald.pval:.3f}")
print(f"Largest lead in absolute value: {coef.loc[:-2, 'beta'].abs().max():.2f}")
print("Lags:", coef.loc[0:, "beta"].round(2).to_dict())

## 24.9 Placebo tests

In [ ]:
def did_estimate(df, cut_month, treated_items, controls=control_items):
    """TWFE DiD with item and month effects; SEs clustered by item."""
    cut_month = pd.Timestamp(cut_month)
    start = cut_month - pd.DateOffset(months=12)
    end = cut_month + pd.DateOffset(months=3)
    d = df[(df["date"] >= start) & (df["date"] <= end)
           & (df["date"] != cut_month)
           & df["item"].isin(list(treated_items) + list(controls))].copy()
    d["tp"] = (d["item"].isin(treated_items) & (d["date"] > cut_month)).astype(int)
    d = d.set_index(["item", "date"])
    r = PanelOLS(d["lp"], d[["tp"]], entity_effects=True,
                 time_effects=True).fit(cov_type="clustered", cluster_entity=True)
    return r.params["tp"], r.std_errors["tp"]


full_panel = cpi[cpi["item"].isin(list(TREATED) + control_items + food_controls)].copy()
full_panel["lp"] = 100 * np.log(full_panel["index"])

rows = {}
for label, month in [("Actual: Sep 2025", "2025-09-01"),
                     ("Placebo: Sep 2024", "2024-09-01"),
                     ("Placebo: Sep 2023", "2023-09-01")]:
    rows[label] = did_estimate(full_panel, month, list(TREATED))
placebo = pd.DataFrame(rows, index=["DiD", "SE"]).T
placebo["t"] = placebo["DiD"] / placebo["SE"]
placebo.round(2)

In [ ]:
rows = {}
for label, month in [("Actual: Sep 2025", "2025-09-01"),
                     ("Placebo: Sep 2024", "2024-09-01"),
                     ("Placebo: Sep 2023", "2023-09-01")]:
    rows[label] = did_estimate(full_panel, month, list(TREATED),
                               controls=food_controls)
food_placebo = pd.DataFrame(rows, index=["DiD", "SE"]).T
food_placebo["t"] = food_placebo["DiD"] / food_placebo["SE"]
food_placebo.round(2)

In [ ]:
oils = ["Mustard Oil", "Refined Oil [sunflower, Soyabean, Saffola, Etc.]",
        "Coconut Oil", "Groundnut Oil"]
w_oil = full_panel[full_panel["item"].isin(oils)].pivot(index="date",
                                                        columns="item", values="lp")
print("Edible oil prices, change Aug 2024 → Dec 2024 (log points):")
print((w_oil.loc["2024-12-01"] - w_oil.loc["2024-08-01"]).round(1).to_string())

In [ ]:
rng = np.random.default_rng(24)
ctrl_panel = full_panel[full_panel["item"].isin(control_items)]
n_fake = min(len(TREATED), len(control_items) // 2)
fake = np.array([
    did_estimate(ctrl_panel, "2025-09-01",
                 list(rng.choice(control_items, n_fake, replace=False)))[0]
    for _ in range(300)])
p_perm = np.mean(np.abs(fake) >= abs(b))
print(f"{n_fake} fake-treated items per draw; placebo DiDs range "
      f"{fake.min():.2f} to {fake.max():.2f}")
print(f"Share of placebo |DiD| at least as large as the actual {b:.2f}: {p_perm:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(fake, bins=30, color="#9db7d5", edgecolor="white")
ax.axvline(b, color="#1f4e79", lw=2)
ax.text(b, ax.get_ylim()[1] * 0.9, f"  actual DiD {b:.2f}", color="#1f4e79",
        fontsize=9)
ax.set_xlabel("DiD estimate, log points")
ax.set_ylabel("Number of placebo draws")
ax.set_title("Placebo: random 'treated' groups drawn from control items",
             loc="left", fontsize=12)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(rect=(0, 0.05, 1, 1))
fig.text(0.01, 0.005, "300 draws, same date (Sep 2025), controls only. Source: "
         "author's calculation from MoSPI CPI item indices.",
         fontsize=8, color="grey")
plt.show()

In [ ]:
general = mospi_cpi("combined").set_index("date")["index"]
yoy = 100 * (general / general.shift(12) - 1)
print(pd.DataFrame({"ours": yoy.loc["2025-08":"2025-12"].round(2)}).T)
print("MoSPI press release, Oct 2025 (provisional): 0.25")

## 24.10 Threats to the design

## 24.11 Regression discontinuity: what happens at 60?

In [ ]:
from indiadata import labour_status, read_plfs

p = read_plfs("perv1", columns=["sex", "age", "acws", "pas"])
men = p[(p["sex"] == 1) & p["age"].between(45, 75)].copy()
del p
men["age"] = men["age"].astype(int)
men["regular"] = (men["acws"] == 31).astype(float)
men["in_lf"] = (labour_status(men["acws"], weekly=True) != "out of LF").astype(float)
men["pensioner"] = (men["pas"] == 94).astype(float)
print(f"{len(men):,} men aged 45–75 in the sample")
men[["age", "acws", "regular", "in_lf"]].head()

### Age heaping

In [ ]:
counts = men.groupby("age").size()
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.bar(counts.index, counts,
       color=["#c55a11" if a % 5 == 0 else "#9db7d5" for a in counts.index])
ax.set_xlabel("Reported age")
ax.set_ylabel("Men in the sample")
ax.set_title("Age heaping: far more men report ages ending in 0 or 5",
             loc="left", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(rect=(0, 0.05, 1, 1))
fig.text(0.01, 0.005, "Unweighted counts. Source: author's calculation from "
         "PLFS 2023–24 unit-level data (NSO, MoSPI).", fontsize=8, color="grey")
plt.show()

ratio_60 = counts[60] / ((counts[59] + counts[61]) / 2)
print(f"Men reporting age 60: {counts[60]:,}; average of ages 59 and 61: "
      f"{(counts[59] + counts[61]) / 2:,.0f}; ratio {ratio_60:.1f}")

In [ ]:
ages = read_plfs("perv1", columns=["sex", "age"])
ages = ages.loc[(ages["sex"] == 1) & ages["age"].between(23, 62), "age"].astype(int)
whipple = 100 * (ages % 5 == 0).sum() / (len(ages) / 5)
print(f"Whipple's index, men, ages 23–62 (unweighted): {whipple:.0f}")
del ages

In [ ]:
wmean = (men.groupby("age")
            .apply(lambda d: np.average(d["regular"], weights=d["weight"]))
            .rename("regular"))
heap_ages = [50, 55, 65, 70]
heap = pd.DataFrame({
    "at the heap": [wmean[a] for a in heap_ages],
    "average of a−1 and a+1": [(wmean[a - 1] + wmean[a + 1]) / 2 for a in heap_ages]},
    index=pd.Index(heap_ages, name="age a"))
(100 * heap).round(1)

### The binned scatter

In [ ]:
def rd_fit(df, y, h, donut=0, drop_heaps=False, cutoff=60):
    """Local linear RD at `cutoff` with bandwidth h (years), survey weights,
    uniform kernel, and standard errors clustered by first-stage unit."""
    d = df[(df["age"] - cutoff).abs() <= h].copy()
    d = d[(d["age"] - cutoff).abs() >= donut] if donut else d
    if drop_heaps:
        d = d[d["age"] % 5 != 0]
    d["x"] = d["age"] - cutoff
    d["above"] = (d["x"] >= 0).astype(int)
    m = smf.wls(f"{y} ~ above + x + above:x", data=d, weights=d["weight"]).fit(
        cov_type="cluster", cov_kwds={"groups": d["mfsu"].astype(int)})
    return m


lfw = (men.groupby("age")
          .apply(lambda d: np.average(d["in_lf"], weights=d["weight"]))
          .rename("in_lf"))
base_fit = rd_fit(men, "regular", h=10, donut=1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
for ax, (y, series, title) in zip(axes, [
        ("regular", wmean, "Regular wage/salaried (CWS)"),
        ("in_lf", lfw, "In the labour force (CWS)")]):
    fit = rd_fit(men, y, h=10, donut=1)
    ax.scatter(series.index, 100 * series,
               c=["#c55a11" if a % 5 == 0 else "#1f4e79" for a in series.index],
               s=22, zorder=3)
    for side in (np.arange(50, 60), np.arange(60, 71)):
        xx = pd.DataFrame({"x": side - 60, "above": int(side[0] >= 60)})
        ax.plot(side, 100 * fit.predict(xx), color="#1f4e79", lw=1.5)
    ax.axvline(59.5, color="grey", ls="--", lw=0.8)
    ax.set_title(title, loc="left", fontsize=11)
    ax.set_xlabel("Age")
    ax.set_ylabel("% of men")
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("Men's employment at the retirement age of 60", x=0.01,
             ha="left", fontsize=13)
plt.tight_layout(rect=(0, 0.06, 1, 0.98))
fig.text(0.01, 0.005, "Dots: weighted share at each age (orange = age ending "
         "in 0 or 5). Lines: local linear fits, ages 50–70, age 60 dropped "
         "(donut). Source: author's calculation from PLFS 2023–24 (NSO, MoSPI).",
         fontsize=8, color="grey")
plt.show()

### Estimates and bandwidth sensitivity

In [ ]:
rows = []
for h in (3, 4, 5, 6, 8, 10):
    for label, kw in [("all ages", {}), ("donut: drop 60", {"donut": 1}),
                      ("drop all heaps", {"drop_heaps": True})]:
        m = rd_fit(men, "regular", h=h, **kw)
        rows.append({"bandwidth": h, "sample": label,
                     "jump (pp)": 100 * m.params["above"],
                     "SE": 100 * m.bse["above"], "n": int(m.nobs)})
rd_table = pd.DataFrame(rows).pivot(index="bandwidth", columns="sample",
                                    values=["jump (pp)", "SE"])
rd_table.round(1)

In [ ]:
main = rd_fit(men, "regular", h=5, donut=1)
lf_main = rd_fit(men, "in_lf", h=5, donut=1)
pen = rd_fit(men, "pensioner", h=5, donut=1)
below = np.average(men.loc[men["age"].between(55, 59), "regular"],
                   weights=men.loc[men["age"].between(55, 59), "weight"])
print(f"Regular salaried, jump at 60: {100 * main.params['above']:.1f} pp "
      f"(SE {100 * main.bse['above']:.1f}); share at 55–59: {100 * below:.1f}%")
print(f"In labour force, jump at 60:  {100 * lf_main.params['above']:.1f} pp "
      f"(SE {100 * lf_main.bse['above']:.1f})")
print(f"Principal status 'pensioner', jump at 60: "
      f"{100 * pen.params['above']:.1f} pp (SE {100 * pen.bse['above']:.1f})")

### Is the jump really about the retirement rule?

In [ ]:
rows = []
for c in [*range(50, 58), 60, *range(63, 71)]:
    sample = men if c == 60 else (men[men["age"] < 60] if c < 60
                                  else men[men["age"] > 60])
    m = rd_fit(sample, "regular", h=4, donut=1, cutoff=c)
    rows.append({"cutoff": c, "jump": 100 * m.params["above"],
                 "se": 100 * m.bse["above"]})
plac = pd.DataFrame(rows).set_index("cutoff")

fig, ax = plt.subplots(figsize=(9, 3.8))
cols = ["#c55a11" if c == 60 else ("#9db7d5" if c % 5 else "#7f7f7f")
        for c in plac.index]
ax.bar(plac.index, plac["jump"], color=cols, yerr=1.96 * plac["se"],
       error_kw={"lw": 0.8, "ecolor": "#555555"})
ax.axhline(0, color="black", lw=0.6)
ax.set_xticks(plac.index)
ax.set_xlabel("Cutoff age used in the RD")
ax.set_ylabel("Jump in regular salaried share, pp")
ax.set_title("Placebo cutoffs: the fall at 60 stands out", loc="left",
             fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(rect=(0, 0.06, 1, 1))
fig.text(0.01, 0.005, "Local linear RD, bandwidth 4, cutoff age dropped; grey "
         "bars = heaped ages (multiples of 5); 95% intervals. Source: author's "
         "calculation from PLFS 2023–24 (NSO, MoSPI).", fontsize=8, color="grey")
plt.show()

big = plac.drop(60)["jump"].abs()
print(f"Jump at 60: {plac.loc[60, 'jump']:.1f} pp. Largest placebo jump: "
      f"{plac.drop(60).loc[big.idxmax(), 'jump']:.1f} pp at age {big.idxmax()}")
print("Cutoffs next to the heap at 55:", plac.loc[[54, 55, 56], "jump"].round(1).to_dict())

## 24.12 Instrumental variables, by simulation

In [ ]:
rng = np.random.default_rng(2024)
n = 5_000
TRUE_RETURN = 0.08                              # chosen, not estimated
ability = rng.normal(0, 1, n)                   # unobserved by the analyst
near_school = rng.binomial(1, 0.5, n)           # the instrument Z
school = np.clip(np.round(8 + 2.0 * near_school + 1.5 * ability
                          + rng.normal(0, 2, n)), 0, 18)
lwage = 9 + TRUE_RETURN * school + 0.15 * ability + rng.normal(0, 0.4, n)
sim = pd.DataFrame({"lwage": lwage, "school": school, "Z": near_school})

ols = smf.ols("lwage ~ school", sim).fit()
print(f"True return {TRUE_RETURN:.3f}; OLS estimate {ols.params['school']:.3f} "
      f"(biased upwards by ability)")

In [ ]:
first = smf.ols("school ~ Z", sim).fit()
reduced = smf.ols("lwage ~ Z", sim).fit()
wald_iv = reduced.params["Z"] / first.params["Z"]

sim["school_hat"] = first.fittedvalues
second = smf.ols("lwage ~ school_hat", sim).fit()
cov_iv = np.cov(sim["Z"], sim["lwage"])[0, 1] / np.cov(sim["Z"], sim["school"])[0, 1]

print(f"First stage: Z raises schooling by {first.params['Z']:.2f} years "
      f"(F = {first.tvalues['Z'] ** 2:.0f})")
print(f"Wald (reduced form / first stage): {wald_iv:.3f}")
print(f"2SLS by hand, second stage:        {second.params['school_hat']:.3f}  "
      f"(SE {second.bse['school_hat']:.4f}: WRONG, see below)")
print(f"Cov(Z,y)/Cov(Z,S):                 {cov_iv:.3f}")

In [ ]:
from linearmodels.iv import IV2SLS

iv = IV2SLS.from_formula("lwage ~ 1 + [school ~ Z]", sim).fit(cov_type="robust")
print(f"IV2SLS: {iv.params['school']:.3f}  (robust SE {iv.std_errors['school']:.4f})")
print(f"First-stage F statistic: {iv.first_stage.diagnostics['f.stat'].iloc[0]:.0f}")

In [ ]:
def one_draw(strength, n=1_000):
    a = rng.normal(0, 1, n)
    z = rng.binomial(1, 0.5, n)
    s = 8 + strength * z + 1.5 * a + rng.normal(0, 2, n)
    y = 9 + TRUE_RETURN * s + 0.15 * a + rng.normal(0, 0.4, n)
    return np.cov(z, y)[0, 1] / np.cov(z, s)[0, 1]


draws = {f"strong (first stage {s} years)" if s > 1 else
         f"weak (first stage {s} years)": np.array([one_draw(s) for _ in range(500)])
         for s in (2.0, 0.2)}
summary = pd.DataFrame({k: {"median": np.median(v),
                            "5th pct": np.percentile(v, 5),
                            "95th pct": np.percentile(v, 95)}
                        for k, v in draws.items()}).T
summary.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bins = np.linspace(-0.3, 0.5, 60)
for (label, v), col in zip(draws.items(), ["#1f4e79", "#c55a11"]):
    ax.hist(np.clip(v, -0.3, 0.5), bins=bins, alpha=0.6, color=col, label=label)
ax.axvline(TRUE_RETURN, color="black", lw=1)
ax.text(TRUE_RETURN, ax.get_ylim()[1] * 0.92, "  true return", fontsize=9)
ax.set_xlabel("IV estimate of the return to schooling (clipped to [−0.3, 0.5])")
ax.set_ylabel("Simulations")
ax.set_title("A weak instrument gives wildly dispersed IV estimates",
             loc="left", fontsize=12)
ax.legend(frameon=False, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(rect=(0, 0.05, 1, 1))
fig.text(0.01, 0.005, "SIMULATED data (500 draws of n = 1,000 each); "
         "parameter values are illustrative.", fontsize=8, color="grey")
plt.show()

## 24.13 Answering the question

In [ ]:
print(f"DiD, all treated items (TWFE, Oct–Dec 2025): {b:.2f} log points "
      f"(SE {se:.2f})")
print(f"Pass-through elasticity: {rho:.2f} (SE {rho_se:.2f})")
print("Median pass-through by rate change:",
      by_rate["pass-through"].round(2).to_dict())
print(f"Placebo Sep 2024 DiD: {placebo.loc['Placebo: Sep 2024', 'DiD']:.2f} "
      f"(SE {placebo.loc['Placebo: Sep 2024', 'SE']:.2f})")

## Key ideas

## Exercises

## Data citation and terms of use

## Further reading